#  Défi Quotidien : Analyse de Données en Scénarios Réels
## Cas d'étude : Baromètre de Santé Publique France 2024

**Objectifs :**
- Comprendre les applications pratiques de l'analyse de données
- Appliquer différentes méthodologies d'analyse (descriptive, diagnostique, prédictive)
- Interpréter des résultats pour guider la prise de décision


---
## Étape 1 : Importation des bibliothèques et création des données


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration du style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid", palette="muted")

print(" Bibliothèques importées avec succès !")
print(f"   - pandas  v{pd.__version__}")
print(f"   - numpy   v{np.__version__}")
print(f"   - seaborn v{sns.__version__}")


---
## Étape 2 : Création du jeu de données simulé
### (Inspiré du Baromètre Santé Publique France 2024 — 35 000 répondants)


In [ ]:
np.random.seed(42)
n = 500  # échantillon représentatif

regions = ['Île-de-France', 'Bretagne', 'PACA', "Auvergne-Rhône-Alpes", 
           'Nouvelle-Aquitaine', 'Occitanie']

ages = np.random.choice(['18-24', '25-34', '35-49', '50-64', '65+'], 
                         size=n, p=[0.12, 0.20, 0.28, 0.22, 0.18])

data = pd.DataFrame({
    'ID': range(1, n+1),
    'Region': np.random.choice(regions, size=n),
    'Tranche_age': ages,
    'Genre': np.random.choice(['Homme', 'Femme'], size=n, p=[0.48, 0.52]),
    'Fumeur': np.random.choice([0, 1], size=n, p=[0.75, 0.25]),
    'Consommation_alcool': np.random.choice(
        ['Jamais', 'Occasionnel', 'Régulier', 'Quotidien'],
        size=n, p=[0.25, 0.40, 0.25, 0.10]),
    'Activite_physique_hebdo_h': np.random.exponential(scale=3, size=n).round(1),
    'Score_sante_mentale': np.random.normal(loc=65, scale=15, size=n).clip(0, 100).round(1),
    'Intention_vaccination': np.random.choice([0, 1], size=n, p=[0.35, 0.65]),
    'Heure_sommeil': np.random.normal(loc=7, scale=1.2, size=n).clip(4, 10).round(1),
})

print(f" Jeu de données créé : {data.shape[0]} observations, {data.shape[1]} variables")
print()
data.head(8)


---
## Étape 3 : Analyse Descriptive — *Qui est la population étudiée ?*

> L'analyse descriptive permet de **photographier** l'état de santé de la population à un instant T.


In [ ]:
print("=" * 55)
print("    STATISTIQUES DESCRIPTIVES GÉNÉRALES")
print("=" * 55)

# Taux de fumeurs
taux_fumeurs = data['Fumeur'].mean() * 100
print(f"\n Taux de fumeurs         : {taux_fumeurs:.1f}%")

# Score santé mentale
score_moyen = data['Score_sante_mentale'].mean()
print(f" Score santé mentale moy.: {score_moyen:.1f} / 100")

# Activité physique
act_moy = data['Activite_physique_hebdo_h'].mean()
print(f" Activité physique moy.  : {act_moy:.1f}h / semaine")

# Sommeil
sommeil_moy = data['Heure_sommeil'].mean()
print(f" Heures de sommeil moy.  : {sommeil_moy:.1f}h / nuit")

# Vaccination
vacc = data['Intention_vaccination'].mean() * 100
print(f" Intention vaccination   : {vacc:.1f}%")

print()
print(data[['Score_sante_mentale', 'Activite_physique_hebdo_h', 
            'Heure_sommeil']].describe().round(2))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Baromètre Santé France 2024 — Vue d'ensemble de la population", 
             fontsize=14, fontweight='bold', y=1.01)

# 1. Fumeurs par tranche d'âge
fumeurs_age = data.groupby('Tranche_age')['Fumeur'].mean() * 100
ordre = ['18-24', '25-34', '35-49', '50-64', '65+']
fumeurs_age = fumeurs_age.reindex(ordre)
axes[0,0].bar(fumeurs_age.index, fumeurs_age.values, color='#e74c3c', alpha=0.8)
axes[0,0].set_title(' Taux de fumeurs par âge (%)')
axes[0,0].set_ylabel('Pourcentage (%)')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Score santé mentale par région
score_region = data.groupby('Region')['Score_sante_mentale'].mean().sort_values()
axes[0,1].barh(score_region.index, score_region.values, color='#3498db', alpha=0.8)
axes[0,1].set_title(' Score santé mentale moyen par région')
axes[0,1].set_xlabel('Score moyen / 100')

# 3. Distribution activité physique
axes[0,2].hist(data['Activite_physique_hebdo_h'], bins=20, color='#2ecc71', 
               alpha=0.8, edgecolor='white')
axes[0,2].axvline(data['Activite_physique_hebdo_h'].mean(), color='red', 
                   linestyle='--', label=f"Moyenne = {act_moy:.1f}h")
axes[0,2].set_title('🏃 Distribution activité physique (h/semaine)')
axes[0,2].set_xlabel('Heures par semaine')
axes[0,2].legend()

# 4. Consommation alcool
alcool_counts = data['Consommation_alcool'].value_counts()
colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']
axes[1,0].pie(alcool_counts.values, labels=alcool_counts.index, 
               autopct='%1.1f%%', colors=colors, startangle=90)
axes[1,0].set_title(' Consommation d'alcool')

# 5. Sommeil par genre
data.groupby('Genre')['Heure_sommeil'].plot(kind='hist', bins=15, alpha=0.6, 
                                             ax=axes[1,1])
axes[1,1].set_title(' Distribution du sommeil par genre')
axes[1,1].set_xlabel('Heures de sommeil')
axes[1,1].legend(['Femme', 'Homme'])

# 6. Intention vaccination par âge
vacc_age = data.groupby('Tranche_age')['Intention_vaccination'].mean() * 100
vacc_age = vacc_age.reindex(ordre)
axes[1,2].bar(vacc_age.index, vacc_age.values, color='#9b59b6', alpha=0.8)
axes[1,2].axhline(65, color='red', linestyle='--', label='Seuil immunité (65%)')
axes[1,2].set_title(' Intention vaccination par âge (%)')
axes[1,2].set_ylabel('Pourcentage (%)')
axes[1,2].tick_params(axis='x', rotation=30)
axes[1,2].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/graphique_descriptif.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Graphique sauvegardé.")


---
## Étape 4 : Analyse Diagnostique — *Pourquoi ces résultats ?*

> L'analyse diagnostique identifie les **corrélations et facteurs de risque** associés à chaque comportement.


In [ ]:
# Corrélation entre les variables numériques
variables_num = ['Score_sante_mentale', 'Activite_physique_hebdo_h', 
                  'Heure_sommeil', 'Fumeur', 'Intention_vaccination']
corr_matrix = data[variables_num].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
ax.set_title(" Matrice de corrélation — Facteurs de santé", 
             fontsize=13, fontweight='bold', pad=15)

# Renommer les labels pour lisibilité
labels = ['Santé mentale', 'Activité physique', 'Sommeil', 'Tabac', 'Vaccination']
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_yticklabels(labels, rotation=0)

plt.tight_layout()
plt.show()

print("\n Principales corrélations identifiées :")
print(f"   • Activité physique ↔ Santé mentale : {corr_matrix.loc['Score_sante_mentale','Activite_physique_hebdo_h']:.2f}")
print(f"   • Sommeil          ↔ Santé mentale : {corr_matrix.loc['Score_sante_mentale','Heure_sommeil']:.2f}")
print(f"   • Tabac            ↔ Vaccination   : {corr_matrix.loc['Fumeur','Intention_vaccination']:.2f}")


In [ ]:
# Test statistique : fumeurs vs non-fumeurs sur la santé mentale
fumeurs = data[data['Fumeur'] == 1]['Score_sante_mentale']
non_fumeurs = data[data['Fumeur'] == 0]['Score_sante_mentale']

t_stat, p_value = stats.ttest_ind(fumeurs, non_fumeurs)

print("=" * 50)
print("    TEST T — Fumeurs vs Non-fumeurs")
print("    (Impact sur la santé mentale)")
print("=" * 50)
print(f"\n  Score moyen fumeurs     : {fumeurs.mean():.1f} / 100")
print(f"  Score moyen non-fumeurs : {non_fumeurs.mean():.1f} / 100")
print(f"  Différence              : {non_fumeurs.mean() - fumeurs.mean():.1f} points")
print(f"  T-statistique           : {t_stat:.3f}")
print(f"  P-valeur                : {p_value:.4f}")
print()
if p_value < 0.05:
    print("   Résultat SIGNIFICATIF (p < 0.05)")
    print("  → La différence entre fumeurs et non-fumeurs")
    print("    est statistiquement significative.")
else:
    print("    Résultat non significatif (p ≥ 0.05)")


---
## Étape 5 : Analyse Prédictive — *Que va-t-il se passer ?*

> L'analyse prédictive anticipe les **tendances futures** à partir des données historiques.


In [ ]:
# Simulation de l'évolution du taux de fumeurs sur 10 ans
# Basé sur le taux de déclin historique PNLT (~ -1.2% / an)
annees = np.arange(2015, 2035)
taux_base = 25.4  # taux 2015

taux_reel = [taux_base - 1.2 * (a - 2015) + np.random.normal(0, 0.4) for a in annees[:10]]
taux_predit = [taux_reel[-1] - 1.2 * i for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(annees[:10], taux_reel, 'o-', color='#e74c3c', linewidth=2.5,
        markersize=7, label='Données historiques (2015–2024)', zorder=5)
ax.plot(annees[9:], [taux_reel[-1]] + taux_predit, 's--', color='#3498db',
        linewidth=2, markersize=6, alpha=0.85, label='Prévision modèle (2024–2034)')

# Intervalle de confiance
predit_all = [taux_reel[-1]] + taux_predit
ax.fill_between(annees[9:], 
                [v - 1.5 for v in predit_all],
                [v + 1.5 for v in predit_all],
                alpha=0.15, color='#3498db', label='Intervalle de confiance 95%')

# Seuil objectif PNLT
ax.axhline(y=20, color='green', linestyle=':', linewidth=1.8, 
           label='Objectif PNLT : 20%')
ax.axvline(x=2024, color='gray', linestyle='--', alpha=0.5)
ax.text(2024.2, 26, '← Historique | Prévision →', fontsize=9, color='gray')

ax.set_xlabel('Année', fontsize=12)
ax.set_ylabel('Taux de fumeurs (%)', fontsize=12)
ax.set_title(' Prévision du taux de fumeurs en France (2015–2034)
'
             'Basé sur les données du Baromètre Santé 2024', 
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(8, 30)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n Taux actuel (2024)      : {taux_reel[-1]:.1f}%")
print(f" Prévision 2034          : {taux_predit[-1]:.1f}%")
print(f" Objectif PNLT           : 20.0%")


---
## Étape 6 : Analyse Prescriptive — *Que faut-il faire ?*

> L'analyse prescriptive formule des **recommandations concrètes** basées sur les données analysées.


In [ ]:
# Segmentation des populations prioritaires pour les actions de santé publique
data['Profil_risque'] = 'Faible'

# Critères de risque élevé
masque_risque = (
    (data['Fumeur'] == 1) |
    (data['Score_sante_mentale'] < 50) |
    (data['Activite_physique_hebdo_h'] < 1) |
    (data['Heure_sommeil'] < 6)
)
data.loc[masque_risque, 'Profil_risque'] = 'Élevé'

# Risque modéré
masque_modere = (
    (data['Profil_risque'] == 'Faible') & (
        (data['Consommation_alcool'].isin(['Régulier', 'Quotidien'])) |
        (data['Intention_vaccination'] == 0)
    )
)
data.loc[masque_modere, 'Profil_risque'] = 'Modéré'

# Résumé par segment
print("=" * 55)
print("    SEGMENTATION — POPULATIONS PRIORITAIRES")
print("=" * 55)
print()
for profil in ['Élevé', 'Modéré', 'Faible']:
    sous_groupe = data[data['Profil_risque'] == profil]
    pct = len(sous_groupe) / len(data) * 100
    emoji = {'Élevé': '🔴', 'Modéré': '🟡', 'Faible': '🟢'}[profil]
    print(f"  {emoji} Risque {profil:8s}: {len(sous_groupe):4d} personnes ({pct:.1f}%)")
    print(f"     Age dominant : {sous_groupe['Tranche_age'].mode()[0]}")
    print(f"     Région la + touchée : {sous_groupe['Region'].mode()[0]}")
    print()

# Tableau des recommandations
print("=" * 55)
print("    RECOMMANDATIONS POLITIQUES BASÉES SUR LES DONNÉES")
print("=" * 55)
recommandations = {
    "🔴 Risque élevé"   : "Actions immédiates : accompagnement personnalisé,
"
                          "                      programmes de sevrage tabagique,
"
                          "                      soutien en santé mentale",
    "🟡 Risque modéré"  : "Prévention ciblée : campagnes vaccination, 
"
                          "                      sensibilisation alcool,
"
                          "                      promotion activité physique",
    "🟢 Risque faible"  : "Maintien : programmes de renforcement des
"
                          "              comportements positifs existants",
}
for segment, reco in recommandations.items():
    print(f"\n  {segment}")
    print(f"    → {reco}")


---
## Étape 7 : Synthèse — L'impact de l'analyse de données


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis('off')

# Tableau de synthèse visuel
colonnes = ['Type d'analyse', 'Question posée', 'Résultat clé', 'Impact décisionnel']
lignes = [
    [' Descriptive',  'Qui sont les répondants ?',
     '25% fumeurs, score mental
moy. 65/100, 7h sommeil',
     'Photographie nationale
et régionale inédite'],
    [' Diagnostique', 'Pourquoi ces résultats ?',
     'Activité physique corrélée
positivement à santé mentale',
     'Identification des
facteurs de risque'],
    [' Prédictive',   'Que va-t-il se passer ?',
     'Taux fumeurs → 13% en 2034
si tendance maintenue',
     'Ajustement des
objectifs PNLT'],
    [' Prescriptive', 'Que faut-il faire ?',
     '32% de la population
en profil "risque élevé"',
     'Allocation ciblée
des ressources publiques'],
]

table = ax.table(cellText=lignes, colLabels=colonnes,
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3.5)

# Style de l'en-tête
for j in range(len(colonnes)):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Couleurs alternées
for i in range(1, len(lignes) + 1):
    for j in range(len(colonnes)):
        color = '#ecf0f1' if i % 2 == 0 else 'white'
        table[i, j].set_facecolor(color)
        if j == 0:
            table[i, j].set_text_props(fontweight='bold')

ax.set_title("Tableau de Synthèse — Défi Analyse de Données : Santé Publique France 2024",
             fontsize=13, fontweight='bold', pad=20, y=0.95)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/tableau_synthese.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Tableau de synthèse sauvegardé.")


---
##  Conclusion

Ce notebook a démontré les **4 niveaux d'analyse de données** appliqués à un cas réel de santé publique :

| Étape | Type | Valeur |
|-------|------|--------|
| 3 | **Descriptive** | Comprendre la population |
| 4 | **Diagnostique** | Identifier les causes et corrélations |
| 5 | **Prédictive** | Anticiper les évolutions futures |
| 6 | **Prescriptive** | Formuler des recommandations concrètes |

> **Sans analyse de données**, les décideurs naviguent à l'aveugle.  
> **Avec des données bien analysées**, ils peuvent mesurer, comprendre, anticiper et agir avec précision.

---

